In [ ]:
# Install dependencies + download dataset
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("iamtapendu/rsna-pneumonia-processed-dataset")
print(path)

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

In [ ]:
class RSNADataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        # drop duplicate patientIds since some have multiple bounding boxes
        self.df = self.df.drop_duplicates(subset='patientId').reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patient_id = self.df.iloc[idx]['patientId']
        label = self.df.iloc[idx]['Target']
        img_path = os.path.join(self.img_dir, f"{patient_id}.png")
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

In [ ]:
transforms = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=3),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = RSNADataset(
    csv_path=os.path.join(path, "stage2_train_metadata.csv"),
    img_dir=os.path.join(path, "Training/Images"),
    transform = transforms
)

test_dataset = RSNADataset(
    csv_path=os.path.join(path, "stage2_test_metadata.csv"),
    img_dir = os.path.join(path, "Test"),
    transform = transforms
)


train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = torchvision.models.densenet121(weights="IMAGENET1K_V1")

num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 1)

# Freeze feature extractor
for param in model.features.parameters():
    param.requires_grad = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

targets = [full_dataset.df.iloc[i]['Target'] for i in train_dataset.indices]
pos = sum(targets)
neg = len(targets) - pos

pos_weight = torch.tensor([neg / pos], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

model.to(device)

In [ ]:
epochs = 7

for epoch in range(epochs):

    # ===== TRAIN =====
    model.train()
    epoch_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")

    # ===== VALIDATE =====
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()

            all_preds.extend(preds.cpu().numpy().ravel())
            all_labels.extend(labels.cpu().numpy().ravel())
            all_probs.extend(probs.cpu().numpy().ravel())

    val_loss /= len(val_loader)


    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    aucc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn+fp+1e-8)

    print(f"Specificity: {specificity:.4f}")
    print("Confusion Matrix:", cm)
    print(f"Train Loss: {epoch_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")
    print(f"AUC: {aucc:.4f}")

In [ ]:
# Save the model
torch.save(model.state_dict(), "pneumonia_model.pth")
print("Model saved!")